# cadcat data quality walkthrough

Explores the Cal-Adapt `cadcat` archive (WRF + LOCA2-Hybrid) in four stages, cheapest first:

1. **Catalog audit** — vocabulary, path conventions, coverage. No data reads, runs in seconds.
2. **Coverage matrices** — which variables exist at which domain and resolution.
3. **Metadata sweep** — open a representative store per facet combination and check it.
4. **Backend comparison** — diff what `climakitae` shows against what is actually in S3.

Stages 3 and 4 need network access to `cadcat.s3.amazonaws.com`.

In [ ]:
import logging

import pandas as pd

import data_audit
from data_audit import standards as S

logging.basicConfig(level=logging.WARNING)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 110)

data_audit.__version__

## 1. Load the catalog

`load_catalog()` reads the live intake-ESM collection through `climakitae`'s `DataCatalog`, falling back to the snapshot bundled inside the package if S3 is unreachable. Pass `"bundled"` to force offline mode.

In [ ]:
catalog = data_audit.load_catalog()  # or load_catalog("bundled")
catalog = data_audit.filter_catalog(catalog)  # WRF + LOCA2 only

print(f"{len(catalog):,} stores, sourced from the {catalog['catalog_source'].iloc[0]} catalog")
catalog.groupby(["activity_id", "grid_label", "table_id"]).size()

### Is the bundled snapshot current?

A non-empty result means the offline fallback has drifted from what is actually served.

In [ ]:
try:
    drift = data_audit.compare_catalog_sources()
    print(f"{len(drift):,} record(s) differ between live and bundled")
    display(drift["presence"].value_counts() if len(drift) else "snapshot is current")
except Exception as exc:
    print(f"live catalog unavailable: {exc}")

## 2. Catalog audit

Validates every record's facets against the documented vocabulary, rebuilds the store path implied by those facets, and looks for coverage gaps. No data access.

In [ ]:
audit = data_audit.audit_catalog(catalog)

print(data_audit.severity_counts(audit).to_string())
data_audit.summarize_by_code(audit)[["level", "code", "n", "example"]]

Drill into any single code. Group on `code` — never parse `message`, which is free text meant for humans.

In [ ]:
code = "catalog.path.member_level_differs"
hits = audit[audit["code"] == code]

print(f"{len(hits):,} record(s)")
display(hits.groupby(["activity_id", "institution_id"]).size())
hits[["source_id", "experiment_id", "member_id", "variable_id", "expected", "actual"]].head()

## 3. Coverage

Three complementary views: what is documented but entirely missing, what is published at an undocumented resolution, and what exists at some domains but not others.

In [ ]:
print("documented but absent everywhere:")
display(data_audit.documented_but_absent(catalog))

print("published at an undocumented table_id:")
display(data_audit.timescale_mismatch(catalog))

print("present at some grids but not others:")
display(data_audit.grid_coverage_asymmetry(catalog))

In [ ]:
# Variables (rows) x domain/resolution (columns), values are store counts.
# A zero in a column is a gap at that resolution.
data_audit.coverage_matrix(catalog, "WRF")

In [ ]:
data_audit.coverage_matrix(catalog, "LOCA2")

### Facet consistency

The documentation warns that "parameters are not always internally consistent". This quantifies it: a partially-populated facet means a query filtering on it silently drops the blank records.

In [ ]:
data_audit.facet_consistency(catalog)

## 4. Build a sample plan

Metadata is a property of `(activity_id, institution_id, table_id, grid_label, variable_id)`, not of which GCM or ensemble member produced the run — so one store per combination is enough. `experiment_id` is included because expected start/end years are per-experiment.

`n_in_group` records how many stores each sampled row stands for.

In [ ]:
plan = data_audit.build_sample_plan(catalog, per_key=1)

print(f"{len(plan):,} stores to open, standing in for {int(plan['n_in_group'].sum()):,}")
plan.head()

## 5. Sweep via ClimateData

Backend A reaches the data the way an Analytics Engine user does. Start with a small slice — `d03` daily is a good first target.

Nothing here calls `.load()`; stores are opened lazily and only coordinates are read.

In [ ]:
from data_audit import ClimakitaeBackend

small = data_audit.build_sample_plan(
    data_audit.filter_catalog(catalog, grid_labels=["d03"], table_ids=["day"])
).head(20)

findings_ckae = data_audit.sweep(small, ClimakitaeBackend(), max_workers=4)

print(data_audit.severity_counts(findings_ckae).to_string())
data_audit.summarize_by_code(findings_ckae)[["level", "code", "n", "example"]]

In [ ]:
# Worst offenders, one row per dataset
data_audit.summarize_by_dataset(findings_ckae).head(15)

## 6. Sweep via raw Zarr

Backend B bypasses `climakitae` entirely and adds store-level structure that never survives into an xarray object: consolidated metadata, chunk shape and size, dtype, compressor.

In [ ]:
from data_audit import ZarrBackend

findings_zarr = data_audit.sweep(
    small, ZarrBackend(), inspect_store=True, max_workers=4
)

data_audit.summarize_by_code(findings_zarr)[["level", "code", "n", "example"]]

In [ ]:
# Chunking, dtype and shape as reported straight from .zmetadata
findings_zarr[findings_zarr["code"] == "store.array.shape"][
    ["activity_id", "table_id", "grid_label", "variable_id", "actual"]
]

## 7. Compare the two access paths

This is the interesting one. `climakitae` repairs some metadata on read — `add_crs_to_downscaled_data` attaches a CRS to stores that lack one — so anything it repairs is invisible through backend A while remaining a defect in the published artifact.

A `grid_mapping` present in `value_a` and absent in `value_b` means the **store** is missing it.

In [ ]:
diff = data_audit.compare_backends(
    small.head(10), ClimakitaeBackend(), ZarrBackend(), max_workers=4
)

disagree = diff[~diff["agree"]]
print(f"{len(disagree)} disagreement(s) across {len(diff)} comparisons")
display(disagree["field"].value_counts())
disagree[["activity_id", "variable_id", "field", "value_a", "value_b"]].head(20)

## 8. Inspect one dataset by hand

When a finding is surprising, open the store and look.

In [ ]:
row = small.iloc[0].to_dict()
dataset, diagnostics = ZarrBackend().open(row)

print(diagnostics)
dataset

In [ ]:
from data_audit import checks

for finding in checks.check_dataset(dataset, row):
    print(f"{finding.level.label:6s} {finding.code:38s} {finding.message}")

## 9. Find stores missing from the catalog

Walks the bucket looking for Zarr stores, then subtracts what the catalog knows about. Slow — scope it to one prefix.

In [ ]:
backend = ZarrBackend()
on_disk = set(backend.list_stores("cadcat/wrf/ucla/cesm2/historical"))
in_catalog = {
    str(p).replace("s3://", "").rstrip("/") for p in catalog["path"].dropna()
}

orphans = sorted(on_disk - in_catalog)
print(f"{len(orphans)} store(s) on S3 with no catalog record")
orphans[:20]

## 10. Write the report

In [ ]:
all_findings = pd.concat([audit, findings_ckae, findings_zarr], ignore_index=True)

data_audit.write_outputs(
    all_findings,
    outdir="qc_output",
    stem="cadcat",
    formats=("csv", "md", "html"),
    title="cadcat WRF + LOCA2 quality report",
)

## Tuning

Every expectation lives in `standards.py`, so adjusting a threshold is a one-line edit rather than a hunt through check code.

In [ ]:
print("documented time ranges:")
for key, value in S.EXPECTED_TIME_RANGE.items():
    print(f"  {key}: {value}")

print(f"\nnominal resolution (m): {S.NOMINAL_RESOLUTION_M}")
print(f"resolution tolerance:   {S.RESOLUTION_TOLERANCE}")
print(f"time tolerance (years): {S.TIME_RANGE_TOLERANCE_YEARS}")
print(f"\nrequired variable attrs:  {S.REQUIRED_VAR_ATTRS}")
print(f"recommended global attrs: {S.RECOMMENDED_GLOBAL_ATTRS}")

print("\nunit comparison examples:")
for expected, actual in [("K", "K"), ("m s-1", "m/s"), ("mm", "kg m-2"), ("K", "degC")]:
    print(f"  {expected!r:12s} vs {actual!r:10s} -> {S.compare_units(expected, actual)}")